# APIM ❤️ FinOps

## FinOps Framework lab
![flow](../../images/finops-framework.gif)

This playground leverages the [FinOps Framework](https://www.finops.org/framework/) and Azure API Management to control AI costs. It uses the [token limit](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-token-limit-policy) policy for each [product](https://learn.microsoft.com/en-us/azure/api-management/api-management-howto-add-products?tabs=azure-portal&pivots=interactive) and integrates [Azure Monitor alerts](https://learn.microsoft.com/en-us/azure/azure-monitor/alerts/alerts-overview) with [Logic Apps](https://learn.microsoft.com/en-us/azure/azure-monitor/alerts/alerts-logic-apps?tabs=send-email) to automatically disable APIM [subscriptions](https://learn.microsoft.com/en-us/azure/api-management/api-management-subscriptions) that exceed cost quotas.

### Result
![result](result.png)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the OpenAI model and version according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}-V24" # change the name to match your naming style
resource_group_location = "swedencentral"

# Shared APIM instance this lab attaches to (does NOT deploy its own APIM).
shared_apim_name = "apim-shared-pdcibwky2f5ms"
shared_apim_resource_group_name = "rg-shared-apim-gateway-V2"

# Backend registered in the shared APIM for this lab's API. The models below
# are already deployed on this account — this lab does not deploy models.
# "location" is only used locally (Cells below) to query retail pricing —
# CONFIRM this matches the actual region of the shared Foundry account.
aiservices_config = [{"name": "shared-foundry", "endpoint": "https://agents-foundry-tazvvonn4lhea.services.ai.azure.com/", "location": "swedencentral"}]

# ⚠️ PRICING STATUS: RESOLVED, confirmed against live prices.azure.com data
# via the diagnostic cell below Cell 10 (client-side exact-match filter to
# cut through the ~1000 "5.4"-containing meter variants: Batch, cached,
# long-context, "pro" tier, DataZone, etc.). Root cause of the original
# "meter not found" was the old `unitOfMeasure eq '1K'` filter silently
# excluding gpt-5.4 (its real unit is '1M'), not the region.
#   gpt-5.4:      "5.4 inp Gl"      = $2.50/1M  |  "5.4 opt Gl"      = $15.00/1M
#   gpt-5.4-mini: "5.4 mini Inp Gl" = $0.75/1M  |  "5.4 mini Opt Gl" = $4.50/1M
# NOTE the inconsistent real-world casing: base model uses lowercase
# inp/opt, mini uses capitalized Inp/Opt. Cell 13 matches with `==`
# (case-sensitive), so this casing must stay exactly as below.
models_config = [ { "name": "gpt-5.4-mini", "publisher": "OpenAI", "version": "2026-03-17", "sku": "GlobalStandard", "capacity": 20, "inputTokensMeterSku": "5.4 mini Inp Gl", "outputTokensMeterSku": "5.4 mini Opt Gl" }, 
                { "name": "gpt-5.4", "publisher": "OpenAI", "version": "2026-03-05", "sku": "GlobalStandard", "capacity": 20, "inputTokensMeterSku": "5.4 inp Gl", "outputTokensMeterSku": "5.4 opt Gl" },
                { "name": "DeepSeek-V3.2", "publisher": "DeepSeek",  "version": "1", "sku": "GlobalStandard", "capacity": 10, "inputTokensMeterSku": "V3.2 Inp glbl", "outputTokensMeterSku": "V3.2 Outp glbl"} ]

apim_products_config = [{"name": "finops-framework-platinum", "displayName": "FinOps Framework - Platinum", "tpm": 2000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Hourly", "costQuota": 15 },
                    {"name": "finops-framework-gold", "displayName": "FinOps Framework - Gold", "tpm": 1000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Hourly", "costQuota": 10}, 
                    {"name": "finops-framework-silver", "displayName": "FinOps Framework - Silver", "tpm": 500, "tokenQuota": 1000000, "tokenQuotaPeriod": "Hourly", "costQuota": 5}]
apim_users_config = [ ]
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1", "product": "finops-framework-platinum" },
                    {"name": "subscription2", "displayName": "Subscription 2", "product": "finops-framework-gold" },
                    {"name": "subscription3", "displayName": "Subscription 3", "product": "finops-framework-silver" },
                     {"name": "subscription4", "displayName": "Subscription 4", "product": "finops-framework-silver" } ]

inference_api_path = "finops-framework-inference" # path to the inference API in the APIM service
inference_api_type = "AzureOpenAI"  # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2025-03-01-preview"

currency_code = 'USD'

utils.print_ok('Notebook initialized')


✅ Notebook initialized ⌚ 17:15:19.641558 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run("az ad signed-in-user show", "Retrieved az ad signed-in-user", "Failed to get az ad signed-in-user")
if output.success and output.json_data:
    current_user_object_id = output.json_data['id']

    

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 17:15:28.303168 :2s]
👉🏽 Current user: ycure@coem.co
👉🏽 Tenant ID: 0c67c9dd-067e-4ab7-adc3-eac91ce94463
👉🏽 Subscription ID: efbaff8f-21cc-49db-8141-2caaf996decd
⚙️ Running: az ad signed-in-user show 
✅ Retrieved az ad signed-in-user ⌚ 17:15:33.342291 :5s]


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations. 

⚠️ Retry this step if you get deployment error: `workspace not active` 

In [4]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "aiServicesConfig": { "value": aiservices_config },
        "apimUsersConfig": { "value": apim_users_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "apimProductsConfig": { "value": apim_products_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "sharedApimName": { "value": shared_apim_name },
        "sharedApimResourceGroupName": { "value": shared_apim_resource_group_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")


⚙️ Running: az group show --name lab-finops-framework-V24 
👉🏽 Using existing resource group 'lab-finops-framework-V24'
⚙️ Running: az deployment group create --name finops-framework --resource-group lab-finops-framework-V24 --template-file main.bicep --parameters params.json 
✅ Deployment 'finops-framework' succeeded ⌚ 17:23:25.318369 :57s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the required outputs from the Bicep deployment.

In [5]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    pricing_dcr_endpoint = utils.get_deployment_output(output, 'pricingDCREndpoint', 'Pricing DCR Endpoint')
    pricing_dcr_immutable_id = utils.get_deployment_output(output, 'pricingDCRImmutableId', 'Pricing DCR ImmutableId')
    pricing_dcr_stream = utils.get_deployment_output(output, 'pricingDCRStream', 'Pricing DCR Stream')
    subscription_quota_dcr_endpoint = utils.get_deployment_output(output, 'subscriptionQuotaDCREndpoint', 'Subscription Quota DCR Endpoint')
    subscription_quota_dcr_immutable_id = utils.get_deployment_output(output, 'subscriptionQuotaDCRImmutableId', 'Subscription Quota DCR ImmutableId')
    subscription_quota_dcr_stream = utils.get_deployment_output(output, 'subscriptionQuotaDCRStream', 'Subscription Quota DCR Stream')
    
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")


⚙️ Running: az deployment group show --name finops-framework -g lab-finops-framework-V24 
✅ Retrieved deployment: finops-framework ⌚ 17:23:43.962385 :4s]
👉🏽 APIM API Gateway URL: https://apim-shared-pdcibwky2f5ms.azure-api.net
👉🏽 Pricing DCR Endpoint: https://dcr-pricing-re2ftiaqm2z7u-r08t-swedencentral.logs.z1.ingest.monitor.azure.com
👉🏽 Pricing DCR ImmutableId: dcr-d6d80940ca7748a4b27ea9e4785982f1
👉🏽 Pricing DCR Stream: Custom-Json-PRICING_CL
👉🏽 Subscription Quota DCR Endpoint: https://dcr-quota-re2ftiaqm2z7u-v0w5-swedencentral.logs.z1.ingest.monitor.azure.com
👉🏽 Subscription Quota DCR ImmutableId: dcr-b7aedbe286a641f48e8ce4b700e9bebd
👉🏽 Subscription Quota DCR Stream: Custom-Json-SUBSCRIPTION_QUOTA_CL
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****f633
👉🏽 Subscription Name: subscription2
👉🏽 Subscription Key: ****cc25
👉🏽 Subscription Name: subscription3
👉🏽 Subscription Key: ****68ba
👉🏽 Subscription Name: subscription4
👉🏽 Subscription Key: ****3503


<a id='pricing'></a>
### 🔍 Display retail pricing info based on the [pricing API](https://learn.microsoft.com/en-us/rest/api/cost-management/retail-prices/azure-retail-prices)



In [6]:
import requests
from tabulate import tabulate 

# Normalizes retailPrice to "$ per 1,000,000 tokens" regardless of the
# meter's unitOfMeasure. Old (gpt-4.1-era) meters are unitOfMeasure='1K',
# so retailPrice*1000 == $/1M. gpt-5.4 meters are unitOfMeasure='1M', so
# retailPrice is already $/1M (multiplier=1). Confirmed against real
# prices.azure.com data — see notes in Cell 2.
# NOTE: we intentionally do NOT filter unitOfMeasure eq '1K' anymore — that
# filter was silently excluding every gpt-5.4 meter, which was the actual
# root cause of "meter not found" (not a wrong region).
UNIT_MULTIPLIER = {"1K": 1000, "1M": 1}

def price_per_million(item):
    unit = item.get('unitOfMeasure')
    multiplier = UNIT_MULTIPLIER.get(unit)
    if multiplier is None:
        return None  # unknown unit — don't silently guess, surface as None
    return item['retailPrice'] * multiplier

def build_pricing_table(json_data, table_data):
    for item in json_data['Items']:
        price = price_per_million(item)
        table_data.append([item['armRegionName'], item['armSkuName'], item.get('unitOfMeasure'), price])

table_data = []
table_data.append(['Region', 'SKU', 'Unit', 'Retail Price ($/1M tokens)'])
for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']    
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&$filter=serviceName eq 'Foundry Models' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        build_pricing_table(prices_json, table_data)
    print(tabulate(table_data, headers='firstrow', tablefmt='psql'))


+---------------+---------------------------------------------+----------+------------------------------+
| Region        | SKU                                         | Unit     |   Retail Price ($/1M tokens) |
|---------------+---------------------------------------------+----------+------------------------------|
| swedencentral | o3 mini 0131 Batch Outp Data Zone           | 1K       |                      2.42    |
| swedencentral | gpt 4.1 nano cached Inp glbl                | 1K       |                      0.025   |
| swedencentral | gpt-realtime-21 Text opt DZ                 | 1M       |                     26.4     |
| swedencentral | gpt4omini-rt-aud1217 Outp regnl             | 1K       |                     24.2     |
| swedencentral | Model 5 Inp glbl                            | 1K       |                      0.033   |
| swedencentral | 54 pro opt Gl                               | 1M       |                    180       |
| swedencentral | 56 terra ShortCo Inp Std Gl 

In [7]:
# 🔎 Diagnostic: find the real gpt-5.4 / gpt-5.4-mini meter SKUs.
# The 5.4 family has MANY variants sharing the substring '5.4': Batch, cd
# (cached), longco (long context), pro (a pricier reasoning tier), nano,
# mini, pp, crossed with Gl (Global) / Dz (DataZone). A broad contains()
# search returns hundreds of matches and is unusable as-is — so we fetch
# broadly, then filter client-side with an EXACT pattern for the 4 meters
# this lab actually needs: base gpt-5.4 and gpt-5.4-mini, real-time (not
# Batch), Global (not DataZone, since aiservices_config/models_config use
# GlobalStandard), input and output, with no other modifier words.
import re

diag = requests.get(
    "https://prices.azure.com/api/retail/prices?currencyCode='" + currency_code + "'"
    "&$filter=productName eq 'Azure OpenAI GPT5' and contains(tolower(skuName), '5.4')"
)
print("status:", diag.status_code)
items = diag.json().get('Items', []) if diag.status_code == 200 else []
print(f"{len(items)} total matching meter(s) across ALL '5.4' variants (Batch, cached, long-context, pro tier, Dz, Gl, etc.)")

# Exact target: "5.4 inp Gl" / "5.4 opt Gl" / "5.4 mini inp Gl" / "5.4 mini opt Gl"
# (case-insensitive, since casing is inconsistent across the catalog — e.g.
# both "Inp" and "inp" appear for otherwise-identical meter kinds).
target_pattern = re.compile(r'^5\.4(\s+mini)?\s+(inp|opt)\s+Gl$', re.IGNORECASE)
exact_matches = [item for item in items if target_pattern.match(item['skuName'])]

print(f"\n{len(exact_matches)} EXACT target meter(s) (base/mini, input/output, Global, real-time, no other modifiers):")
for item in exact_matches:
    print(f"  skuName='{item['skuName']}' | armRegionName='{item.get('armRegionName')}' | unitOfMeasure='{item.get('unitOfMeasure')}' | retailPrice={item['retailPrice']}")

if len(exact_matches) < 4:
    print("\n⚠️ Fewer than 4 exact matches (need: base inp, base opt, mini inp, mini opt).")
    print("Nearby candidates that reference '5.4'/'5.4 mini' + inp/opt + Global but did NOT match exactly — inspect manually, do not assume one of these is it without checking the modifier words:")
    near_pattern = re.compile(r'^5\.4(\s+mini)?\s+.*(inp|opt).*\bGl$', re.IGNORECASE)
    seen = {id(i) for i in exact_matches}
    for item in items:
        if near_pattern.match(item['skuName']) and id(item) not in seen:
            print(f"  skuName='{item['skuName']}' | armRegionName='{item.get('armRegionName')}' | unitOfMeasure='{item.get('unitOfMeasure')}' | retailPrice={item['retailPrice']}")

print("\nCopy the exact skuName strings from the 'EXACT target meter(s)' list above into")
print("inputTokensMeterSku / outputTokensMeterSku in Cell 2's models_config.")


status: 200
1000 total matching meter(s) across ALL '5.4' variants (Batch, cached, long-context, pro tier, Dz, Gl, etc.)

65 EXACT target meter(s) (base/mini, input/output, Global, real-time, no other modifiers):
  skuName='5.4 opt Gl' | armRegionName='uaenorth' | unitOfMeasure='1M' | retailPrice=15.0
  skuName='5.4 mini Inp Gl' | armRegionName='centralus' | unitOfMeasure='1M' | retailPrice=0.75
  skuName='5.4 mini Opt Gl' | armRegionName='spaincentral' | unitOfMeasure='1M' | retailPrice=4.5
  skuName='5.4 mini Opt Gl' | armRegionName='brazilsouth' | unitOfMeasure='1M' | retailPrice=4.5
  skuName='5.4 mini Inp Gl' | armRegionName='northcentralus' | unitOfMeasure='1M' | retailPrice=0.75
  skuName='5.4 mini Opt Gl' | armRegionName='polandcentral' | unitOfMeasure='1M' | retailPrice=4.5
  skuName='5.4 mini Inp Gl' | armRegionName='spaincentral' | unitOfMeasure='1M' | retailPrice=0.75
  skuName='5.4 opt Gl' | armRegionName='canadaeast' | unitOfMeasure='1M' | retailPrice=15.0
  skuName='5.4 

<a id='4'></a>
### 4️⃣ Load the pricing data into Azure Monitor custom table

👉 This script uses retail price information. Please adjust it to apply a discount or to use a flat rate with PTUs.   
👉 Prices are normalized to $ per 1,000,000 tokens regardless of the meter's `unitOfMeasure` (gpt-4.1-era meters are `'1K'`, gpt-5.4 meters are confirmed `'1M'` — see Cell 2/10 notes). `InputTokensPrice`/`OutputTokensPrice` in the uploaded rows are therefore $/1M tokens.   
👉 Deploy this script as a [job](https://learn.microsoft.com/en-us/azure/container-apps/jobs?tabs=azure-cli) to run automatically on a predefined schedule.


In [8]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

# Same unit normalization as Cell 10 — see notes there and in Cell 2.
UNIT_MULTIPLIER = {"1K": 1000, "1M": 1}

def price_per_million(item):
    unit = item.get('unitOfMeasure')
    multiplier = UNIT_MULTIPLIER.get(unit)
    if multiplier is None:
        return None
    return item['retailPrice'] * multiplier

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=pricing_dcr_endpoint, credential=credential, logging_enable=False)

for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&$filter=serviceName eq 'Foundry Models' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        if prices_json and 'Items' in prices_json:
            for deployment in models_config:
                if not deployment.get("inputTokensMeterSku") or not deployment.get("outputTokensMeterSku"):
                    utils.print_error(f"Skipping model {deployment.get('name')}: inputTokensMeterSku/outputTokensMeterSku not set yet (run the Cell 11 diagnostic and fill these in Cell 2 first)")
                    continue
                input_item = next((item for item in prices_json['Items'] if item.get('skuName') == deployment.get("inputTokensMeterSku")), None)
                output_item = next((item for item in prices_json['Items'] if item.get('skuName') == deployment.get("outputTokensMeterSku")), None)
                input_tokens_price = price_per_million(input_item) if input_item else None
                output_tokens_price = price_per_million(output_item) if output_item else None
                utils.print_info(f"Adding model {deployment.get("name")} with input / output tokens price {input_tokens_price} / {output_tokens_price}")
                body = [{ "TimeGenerated": str(datetime.now(timezone.utc)),
                        "Model": deployment.get("name"),
                        "InputTokensPrice": input_tokens_price,
                        "OutputTokensPrice": output_tokens_price }]
                try:
                    client.upload(rule_id=pricing_dcr_immutable_id, stream_name=pricing_dcr_stream, logs=body)
                    utils.print_ok(f"Upload succeeded for model {deployment.get("name")}")
                except HttpResponseError as e:
                    utils.print_error(f"Upload failed: {e}")            


👉🏽 Adding model gpt-5.4-mini with input / output tokens price 0.75 / 4.5
✅ Upload succeeded for model gpt-5.4-mini ⌚ 17:24:27.454409 
👉🏽 Adding model gpt-5.4 with input / output tokens price None / 15.0
✅ Upload succeeded for model gpt-5.4 ⌚ 17:24:27.763075 
👉🏽 Adding model DeepSeek-V3.2 with input / output tokens price 0.58 / 1.6800000000000002
✅ Upload succeeded for model DeepSeek-V3.2 ⌚ 17:24:28.022616 


<a id='5'></a>
### 5️⃣ Load the Subscription Quota into Azure Monitor custom table


In [9]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=subscription_quota_dcr_endpoint, credential=credential, logging_enable=False)

for subscription in apim_subscriptions_config:
    for product in apim_products_config:
        if product.get("name") == subscription.get("product"):
            cost_quota = product.get("costQuota")
            utils.print_info(f"Adding {subscription.get('name')} with cost quota {cost_quota}")
            body = [{ 
                "TimeGenerated": str(datetime.now(timezone.utc)),
                "Subscription": subscription.get("name"),
                "Email": subscription.get("email"),
                "CostQuota": cost_quota
            }]
            try:
                client.upload(rule_id=subscription_quota_dcr_immutable_id, stream_name=subscription_quota_dcr_stream, logs=body)
                utils.print_ok(f"Upload succeeded for {subscription.get("name")}")
            except HttpResponseError as e:
                utils.print_error(f"Upload failed: {e}")            


👉🏽 Adding subscription1 with cost quota 15
✅ Upload succeeded for subscription1 ⌚ 17:24:45.716122 
👉🏽 Adding subscription2 with cost quota 10
✅ Upload succeeded for subscription2 ⌚ 17:24:46.032599 
👉🏽 Adding subscription3 with cost quota 5
✅ Upload succeeded for subscription3 ⌚ 17:24:46.318280 
👉🏽 Adding subscription4 with cost quota 5
✅ Upload succeeded for subscription4 ⌚ 17:24:46.618722 


<a id='sdk'></a>
### 🧪 Execute multiple runs using the Azure OpenAI Python SDK

👉 We will send requests with random subscription and models. Adjust the `sleep_time_ms` and the number of `runs` to your test scenario.


In [10]:
import time, random
from openai import AzureOpenAI

runs = 10
sleep_time_ms = 100

for i in range(runs):
    apim_subscription = random.choice(apim_subscriptions)
    openai_model = random.choice(models_config)
    client = AzureOpenAI(
        azure_endpoint = f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key = apim_subscription.get("key"),
        api_version = inference_api_version
    )
    try:
        response = client.chat.completions.create(
            model = str(openai_model.get('name')),
            messages = [
                {"role": "user", "content": "Can you tell me the time, please?"}
            ],
            extra_headers = {"x-user-id": "alex"}
        )
        print(f"▶️ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] 💬 {response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] Error: {e}")
    time.sleep(sleep_time_ms/1000)


▶️ Run 1/10: [subscription2 w/ gpt-5.4-mini] 💬 I can’t see your local time directly. If you want, tell me your time zone or city and I can help estimate it, or you can check your device clock.
▶️ Run 2/10: [subscription3 w/ DeepSeek-V3.2] 💬 I don't have access to real-time data or updates, including the current time.  

To check the time, you could:  
- Look at your device’s clock (computer, phone, etc.)  
- Ask a voice assistant if you have one available (e.g., Siri, Google Assistant, Alexa)  
- Search “current time” in a web browser
▶️ Run 3/10: [subscription3 w/ gpt-5.4] 💬 The current date and time is **2026-09-04 04:15:36 UTC**.
▶️ Run 4/10: [subscription4 w/ gpt-5.4-mini] 💬 I can’t see your local clock from here, so I can’t tell the exact current time.

If you want, I can help you find it based on your device, or you can tell me your time zone/city and I can give you the current time there.
▶️ Run 5/10: [subscription3 w/ DeepSeek-V3.2] 💬 I can’t provide the current time for you. M

<a id='workbooks'></a>
### 🔍 Open the dashboard and workbooks in the Azure Portal

👉 The Cost Analysis workbook contains information on the total costs and quotas for each subscription.  
👉 The [Azure OpenAI Insights workbook](https://github.com/dolevshor/Azure-OpenAI-Insights) provides comprehensive details about service and model usage. Credits to [Dolev Shor](https://github.com/dolevshor/Azure-OpenAI-Insights).  
👉 The [Alerts workbook](https://github.com/microsoft/AzureMonitorCommunity/tree/master/Azure%20Services) provides information about the alerts triggered by Azure Monitor.  

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.